# 🎲 Chapter 2: Axioms of Probability
## MAT204 — Probability and Statistics for Engineers

**Topics:**
1. Sample Space and Events
2. Set Theory and Operations (∩, ∪, Aᶜ)
3. DeMorgan's Laws
4. Axioms of Probability (Kolmogorov)
5. Basic Propositions (Complement, Difference, Inclusion-Exclusion)
6. Equally Likely Sample Spaces
7. The Birthday Problem

---

In [ ]:
# Required libraries
import random
import itertools
import math
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle, FancyArrowPatch
from fractions import Fraction

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 100
random.seed(42)
np.random.seed(42)

print("Libraries loaded successfully! ✓")

---
## 1. Sample Space and Events

**Definition:** A **sample space** $S$ is the *set of all possible outcomes* of an experiment.

**Definition:** An **event** $E$ is any subset of the sample space $S$: $E \subseteq S$

- Impossible event: $\emptyset$ (empty set)
- Certain event: $S$ itself

In [ ]:
# Sample space examples — represented using Python sets

# 1) Three coin tosses
S_coins = set(itertools.product(['H','T'], repeat=3))
S_coins_str = {''.join(s) for s in S_coins}
print(f"3-coin-toss sample space S = {sorted(S_coins_str)}")
print(f"  |S| = {len(S_coins_str)}  (2³ = 8 ✓)")

# Event: exactly 2 heads
E_2heads = {s for s in S_coins_str if s.count('H') == 2}
print(f"  Event E (exactly 2 heads): E = {sorted(E_2heads)}, |E| = {len(E_2heads)}")

print()

# 2) One die roll
S_die = {1, 2, 3, 4, 5, 6}
E_even = {x for x in S_die if x % 2 == 0}
E_odd  = {x for x in S_die if x % 2 != 0}
print(f"One die: S = {sorted(S_die)}")
print(f"  E (even) = {sorted(E_even)}")
print(f"  E (odd)  = {sorted(E_odd)}")
print(f"  E(even) ∩ E(odd) = {E_even & E_odd}  ← Mutually exclusive!")

print()

# 3) Sum of two dice
S_two_dice_pairs = [(i, j) for i in range(1, 7) for j in range(1, 7)]
S_two_dice = {i + j for i in range(1, 7) for j in range(1, 7)}  # set of sum values
print(f"Sum of two dice (all pairs): |S| = {len(S_two_dice_pairs)} (6×6 = 36 elements)")
print(f"  Possible sum values: {sorted(S_two_dice)}")

print()

# 4) Three-nucleotide sequences
S_nucleotide = set(itertools.product('ACGT', repeat=3))
print(f"Three nucleotides: |S| = {len(S_nucleotide)} = 4³  (A, C, G, T)")
print(f"  First 8 examples: {sorted([''.join(s) for s in S_nucleotide])[:8]}...")

In [ ]:
# Visualization: distribution of the sum of two dice
outcomes = [(i, j, i+j) for i in range(1,7) for j in range(1,7)]
sums     = [t for _, _, t in outcomes]
freq     = {k: sums.count(k) for k in range(2, 13)}

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: heat map (all pairs)
ax = axes[0]
matrix = np.array([[i+j for j in range(1,7)] for i in range(1,7)])
im = ax.imshow(matrix, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(6)); ax.set_xticklabels(range(1,7))
ax.set_yticks(range(6)); ax.set_yticklabels(range(1,7))
ax.set_xlabel('2nd Die', fontsize=11)
ax.set_ylabel('1st Die', fontsize=11)
ax.set_title('Sum of Two Dice — Sample Space', fontsize=12, fontweight='bold')
for i in range(6):
    for j in range(6):
        ax.text(j, i, matrix[i,j], ha='center', va='center',
                fontsize=11, fontweight='bold',
                color='white' if matrix[i,j] >= 9 else 'black')
plt.colorbar(im, ax=ax, label='Sum')

# Right: probability bar chart
ax2 = axes[1]
x = list(freq.keys())
y = [v/36 for v in freq.values()]
colors = plt.cm.YlOrRd(np.array(y) / max(y))
bars = ax2.bar(x, y, color=colors, edgecolor='black', linewidth=0.7)
ax2.axhline(1/6, color='blue', linestyle='--', alpha=0.5, label='1/6 reference')
for bar, val in zip(bars, y):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
             f'{val:.3f}', ha='center', va='bottom', fontsize=8)
ax2.set_xlabel('Sum', fontsize=11)
ax2.set_ylabel('P(E)', fontsize=11)
ax2.set_title('Probabilities of the Sum of Two Dice', fontsize=12, fontweight='bold')
ax2.set_xticks(x)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()
print(f"Total probability: {sum(y):.4f}  ← P(S) = 1 ✓")

---
## 2. Set Operations

| Operation | Symbol | Meaning |
|-----------|--------|---------|
| Intersection | $A \cap B$ | Both A and B occur |
| Union | $A \cup B$ | A or B (at least one) |
| Complement | $A^c = S \setminus A$ | A does not occur |
| Subset | $A \subseteq B$ | If A occurs, then B occurs |
| Disjoint | $A \cap B = \emptyset$ | A and B are mutually exclusive |

In [ ]:
# Set operations with Python
S = set(range(1, 7))  # Die: {1,2,3,4,5,6}
A = {2, 4, 6}         # Even numbers
B = {1, 2, 3}         # Less than or equal to 3

print(f"S = {sorted(S)}")
print(f"A = {sorted(A)}  (even)")
print(f"B = {sorted(B)}  (≤ 3)")
print()
print(f"A ∩ B = {sorted(A & B)}   (even and ≤ 3)")
print(f"A ∪ B = {sorted(A | B)}  (even or ≤ 3)")
print(f"Aᶜ    = {sorted(S - A)}    (not even = odd)")
print(f"Bᶜ    = {sorted(S - B)}  (greater than 3)")
print(f"A ∩ Bᶜ= {sorted(A - B)}    (even AND greater than 3)")
print()

# Set relations
C = {2, 4}            # Subset of A
print(f"C = {sorted(C)}  (subset of A?): C ⊆ A → {C.issubset(A)}")
print(f"Are A and B disjoint? A ∩ B = ∅ → {len(A & B) == 0}")
D = {1, 3, 5}         # odd numbers
print(f"D = {sorted(D)}  (odd). Are A and D disjoint? → {len(A & D) == 0}  ✓ mutually exclusive")

In [ ]:
# Venn Diagrams
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

def venn2(ax, title, shade_func, A_label='A', B_label='B'):
    ax.set_xlim(0, 10); ax.set_ylim(0, 7)
    ax.set_aspect('equal'); ax.axis('off')
    # S rectangle
    rect = plt.Rectangle((0.2, 0.5), 9.6, 6, fill=False,
                          edgecolor='black', linewidth=2)
    ax.add_patch(rect)
    ax.text(9.5, 6.3, 'S', fontsize=13, fontweight='bold')
    # Circles
    cA = Circle((3.5, 3.5), 2.2, fill=False, edgecolor='#2166ac', linewidth=2)
    cB = Circle((6.5, 3.5), 2.2, fill=False, edgecolor='#d6604d', linewidth=2)
    ax.add_patch(cA); ax.add_patch(cB)
    ax.text(2.2, 3.5, A_label, fontsize=14, fontweight='bold', color='#2166ac', ha='center')
    ax.text(7.8, 3.5, B_label, fontsize=14, fontweight='bold', color='#d6604d', ha='center')
    # Shading
    shade_func(ax)
    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)

from matplotlib.patches import PathPatch
from matplotlib.path import Path

# Left: A ∪ B
def shade_union(ax):
    cA = Circle((3.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=1)
    cB = Circle((6.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=1)
    ax.add_patch(cA); ax.add_patch(cB)
    ax.text(5, 1.0, 'Shaded region: A ∪ B', ha='center', fontsize=9, color='#0571b0')

# Middle: A ∩ B
def shade_intersection(ax):
    theta = np.linspace(0, 2*np.pi, 300)
    # Approximate intersection region
    x1 = 3.5 + 2.2*np.cos(theta)
    y1 = 3.5 + 2.2*np.sin(theta)
    x2 = 6.5 + 2.2*np.cos(theta)
    y2 = 3.5 + 2.2*np.sin(theta)
    # Circle A
    c1 = Circle((3.5, 3.5), 2.2, color='none', ec='#2166ac', lw=2)
    ax.add_patch(c1)
    # Shade only the intersection
    from matplotlib.patches import Wedge
    c_shade = Circle((5, 3.5), 1.1, color='#f4a582', alpha=0.8, zorder=2)
    ax.add_patch(c_shade)
    ax.text(5, 1.0, 'Shaded region: A ∩ B', ha='center', fontsize=9, color='#ca0020')

# Right: Aᶜ
def shade_complement(ax):
    rect = plt.Rectangle((0.2, 0.5), 9.6, 6, color='#b2df8a', alpha=0.5, zorder=1)
    ax.add_patch(rect)
    cA = Circle((3.5, 3.5), 2.2, color='white', zorder=2)
    ax.add_patch(cA)
    ax.text(5, 1.0, 'Shaded region: Aᶜ', ha='center', fontsize=9, color='#33a02c')

venn2(axes[0], 'Union: A ∪ B', shade_union)
venn2(axes[1], 'Intersection: A ∩ B', shade_intersection)
venn2(axes[2], 'Complement: Aᶜ', shade_complement)

plt.suptitle('Set Operations — Venn Diagrams', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### Set Laws — Numerical Verification

In [ ]:
# Numerically verify set laws over a large universe
universe = set(range(1, 21))  # S = {1,...,20}
A = {2,4,6,8,10,12,14,16,18,20}  # even
B = {1,2,3,4,5,6,7,8,9,10}       # ≤ 10
C = {5,6,7,8,9,10,11,12,13,14,15}  # 5–15

laws = [
    ("Commutativity (∪)",    A | B,           B | A),
    ("Commutativity (∩)",    A & B,           B & A),
    ("Associativity (∪)",   (A | B) | C,     A | (B | C)),
    ("Associativity (∩)",   (A & B) & C,     A & (B & C)),
    ("Distributivity 1",    (A | B) & C,     (A & C) | (B & C)),
    ("Distributivity 2",    (A & B) | C,     (A | C) & (B | C)),
    ("DeMorgan 1",           universe-(A | B),   (universe-A) & (universe-B)),
    ("DeMorgan 2",           universe-(A & B),   (universe-A) | (universe-B)),
]

print(f"{'Law':<22} {'Equal?'}")
print("-" * 35)
for name, lhs, rhs in laws:
    equal = lhs == rhs
    print(f"{name:<22} {'✓  Verified' if equal else '✗  ERROR'}")

---
## 3. DeMorgan's Laws

$$(A \cup B)^c = A^c \cap B^c \qquad (A \cap B)^c = A^c \cup B^c$$

Generalized form:
$$\left(\bigcup_{i=1}^n A_i\right)^c = \bigcap_{i=1}^n A_i^c \qquad \left(\bigcap_{i=1}^n A_i\right)^c = \bigcup_{i=1}^n A_i^c$$

**Intuition:** The complement of "AND" becomes "OR"; the complement of "OR" becomes "AND".

In [ ]:
# Demonstrate DeMorgan's Laws with a concrete example
S = set(range(1, 13))  # {1,...,12}
A = {x for x in S if x % 2 == 0}   # even: {2,4,6,8,10,12}
B = {x for x in S if x % 3 == 0}   # multiples of 3: {3,6,9,12}

print(f"S = {{1,...,12}}")
print(f"A = {sorted(A)}  (even)")
print(f"B = {sorted(B)}  (multiples of 3)")
print()

# DeMorgan 1: (A∪B)ᶜ = Aᶜ∩Bᶜ
lhs1 = S - (A | B)
rhs1 = (S - A) & (S - B)
print(f"DeMorgan 1: (A∪B)ᶜ = Aᶜ∩Bᶜ")
print(f"  A ∪ B = {sorted(A|B)}  (even OR multiple of 3)")
print(f"  (A∪B)ᶜ = {sorted(lhs1)}  (neither even nor multiple of 3)")
print(f"  Aᶜ∩Bᶜ  = {sorted(rhs1)}")
print(f"  Equal? {lhs1 == rhs1}  ✓")
print()

# DeMorgan 2: (A∩B)ᶜ = Aᶜ∪Bᶜ
lhs2 = S - (A & B)
rhs2 = (S - A) | (S - B)
print(f"DeMorgan 2: (A∩B)ᶜ = Aᶜ∪Bᶜ")
print(f"  A ∩ B = {sorted(A&B)}  (even AND multiple of 3 = multiples of 6)")
print(f"  (A∩B)ᶜ = {sorted(lhs2)}")
print(f"  Aᶜ∪Bᶜ  = {sorted(rhs2)}")
print(f"  Equal? {lhs2 == rhs2}  ✓")

# Everyday language interpretation
print()
print("Everyday language interpretation:")
print("  DeMorgan 1: 'NOT (even OR multiple of 3)' = 'NOT even AND NOT multiple of 3'")
print("  DeMorgan 2: 'NOT (even AND multiple of 3)' = 'NOT even OR NOT multiple of 3'")

In [ ]:
# DeMorgan's Laws — Venn diagram visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax_idx, (title, law_no) in enumerate([
    ("DeMorgan 1: $(A∪B)^c = A^c \\cap B^c$", 1),
    ("DeMorgan 2: $(A∩B)^c = A^c \\cup B^c$", 2)
]):
    ax = axes[ax_idx]
    ax.set_xlim(0, 10); ax.set_ylim(0, 7)
    ax.set_aspect('equal'); ax.axis('off')
    ax.set_title(title, fontsize=11, fontweight='bold')

    # S rectangle — background
    if law_no == 1:
        bg = plt.Rectangle((0.2, 0.5), 9.6, 6, color='#92c5de', alpha=0.4, zorder=0)
    else:
        bg = plt.Rectangle((0.2, 0.5), 9.6, 6, color='#92c5de', alpha=0.4, zorder=0)
    ax.add_patch(bg)
    rect = plt.Rectangle((0.2, 0.5), 9.6, 6, fill=False, edgecolor='black', linewidth=2)
    ax.add_patch(rect)

    if law_no == 1:
        # Shaded = (A∪B)ᶜ: the rest of S
        cA = Circle((3.5, 3.5), 2.2, color='white', zorder=2)
        cB = Circle((6.5, 3.5), 2.2, color='white', zorder=2)
        ax.add_patch(cA); ax.add_patch(cB)
        ca = Circle((3.5,3.5),2.2,fill=False,ec='#2166ac',lw=2,zorder=3)
        cb = Circle((6.5,3.5),2.2,fill=False,ec='#d6604d',lw=2,zorder=3)
        ax.add_patch(ca); ax.add_patch(cb)
        ax.text(5, 0.8, '(A∪B)ᶜ = Aᶜ∩Bᶜ (blue region)', ha='center', fontsize=9)
    else:
        # Shaded = (A∩B)ᶜ: everything except the intersection
        cA = Circle((3.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=2)
        cB = Circle((6.5, 3.5), 2.2, color='#92c5de', alpha=0.6, zorder=2)
        ax.add_patch(cA); ax.add_patch(cB)
        # Intersection in white (excluded)
        c_int = Circle((5, 3.5), 1.05, color='white', zorder=3)
        ax.add_patch(c_int)
        ca = Circle((3.5,3.5),2.2,fill=False,ec='#2166ac',lw=2,zorder=4)
        cb = Circle((6.5,3.5),2.2,fill=False,ec='#d6604d',lw=2,zorder=4)
        ax.add_patch(ca); ax.add_patch(cb)
        ax.text(5, 0.8, '(A∩B)ᶜ = Aᶜ∪Bᶜ (blue region)', ha='center', fontsize=9)

    ax.text(2.5, 3.5, 'A', fontsize=14, fontweight='bold', color='#2166ac', ha='center', zorder=5)
    ax.text(7.5, 3.5, 'B', fontsize=14, fontweight='bold', color='#d6604d', ha='center', zorder=5)
    ax.text(9.5, 6.3, 'S', fontsize=13, fontweight='bold')

plt.suptitle("DeMorgan's Laws — Venn Diagram Illustration",
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 4. Axioms of Probability (Kolmogorov, 1933)

$P$ is a function that assigns a non-negative real number to each event $E$. $P$ is called a **probability** if:

**Axiom 1 (Non-negativity):**
$$0 \leq P(E) \leq 1$$

**Axiom 2 (Total probability):**
$$P(S) = 1$$

**Axiom 3 (Countable additivity — for mutually exclusive events):**
$$P\!\left(\bigcup_{i=1}^{\infty} E_i\right) = \sum_{i=1}^{\infty} P(E_i), \quad E_i \cap E_j = \emptyset, \ i \neq j$$

In [ ]:
# A probability function class that verifies the axioms
class ProbabilityFunction:
    def __init__(self, S, p_dict):
        """S: sample space (set), p_dict: probability of each outcome"""
        self.S = S
        self.p = p_dict
        self._verify_axioms()

    def _verify_axioms(self):
        # Axiom 1
        assert all(0 <= v <= 1 for v in self.p.values()), "Axiom 1 violated!"
        # Axiom 2
        assert abs(sum(self.p.values()) - 1.0) < 1e-9, "Axiom 2 violated!"
        print("Axiom 1 (0 ≤ P(e) ≤ 1): ✓")
        print(f"Axiom 2 (P(S) = {sum(self.p.values()):.6f}): ✓")

    def P(self, E):
        """Probability of event E"""
        return sum(self.p.get(e, 0) for e in E)

    def P_intersection(self, A, B): return self.P(A & B)
    def P_union(self, A, B):        return self.P(A | B)
    def P_complement(self, A):      return 1 - self.P(A)


# Example 1: Fair die
print("=== Fair Die ===")
S_die = {1, 2, 3, 4, 5, 6}
p_fair = {i: 1/6 for i in S_die}
die = ProbabilityFunction(S_die, p_fair)

E_even    = {2, 4, 6}
E_five_six = {5, 6}
print(f"P(even) = {die.P(E_even):.4f}  = {Fraction(die.P(E_even)).limit_denominator(10)}")
print(f"P(5 or 6) = {die.P(E_five_six):.4f}")
print(f"P(evenᶜ) = P(odd) = {die.P_complement(E_even):.4f}")
print()

# Example 2: Weighted (loaded) die
print("=== Loaded Die (6 is twice as likely) ===")
p_loaded = {1: 1/7, 2: 1/7, 3: 1/7, 4: 1/7, 5: 1/7, 6: 2/7}
die_l = ProbabilityFunction(S_die, p_loaded)
print(f"P(6) = {die_l.P({6}):.4f}  (twice that of a fair die)")
print(f"P(even) = {die_l.P(E_even):.4f}  (fair: 0.5)")

In [ ]:
# Axiom 3 — Countable additivity visualization
# Mutually exclusive events in a die roll
fig, ax = plt.subplots(figsize=(11, 5))

events = [
    ({1}, 'E₁={1}', '#e41a1c'),
    ({2}, 'E₂={2}', '#377eb8'),
    ({3}, 'E₃={3}', '#4daf4a'),
    ({4}, 'E₄={4}', '#984ea3'),
    ({5}, 'E₅={5}', '#ff7f00'),
    ({6}, 'E₆={6}', '#a65628'),
]

x_pos = np.arange(len(events))
probs  = [1/6] * 6
colors = [e[2] for e in events]

bars = ax.bar(x_pos, probs, color=colors, edgecolor='black',
              linewidth=0.8, alpha=0.85, width=0.7)
ax.set_xticks(x_pos)
ax.set_xticklabels([e[1] for e in events], fontsize=11)
ax.set_ylabel('P(Eᵢ)', fontsize=12)
ax.set_ylim(0, 0.8)
ax.set_title('Axiom 3: Sum of Mutually Exclusive Events = P(S) = 1',
             fontsize=13, fontweight='bold')
ax.axhline(sum(probs), color='red', linestyle='--', linewidth=2,
           label=f'P(S) = Σ P(Eᵢ) = {sum(probs):.4f}')

# Cumulative sum annotation
cumulative = 0
for i, p in enumerate(probs):
    ax.text(i, p + 0.02, f'{p:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    cumulative += p

ax.text(5.5, 0.75, f'Total: {sum(probs):.4f}', ha='right', fontsize=12,
        bbox=dict(boxstyle='round', facecolor='lightyellow', edgecolor='orange'))
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

---
## 5. Basic Probability Propositions

| Proposition | Formula | Condition |
|-------------|---------|----------|
| Complement | $P(A^c) = 1 - P(A)$ | — |
| Empty set | $P(\emptyset) = 0$ | — |
| Difference | $P(B \cap A^c) = P(B) - P(A)$ | $A \subseteq B$ |
| Monotonicity | $P(B) \geq P(A)$ | $A \subseteq B$ |
| Inclusion-Exclusion (2) | $P(A \cup B) = P(A) + P(B) - P(A \cap B)$ | — |
| Inclusion-Exclusion (3) | $P(A∪B∪C) = P(A)+P(B)+P(C)-P(A∩B)-P(A∩C)-P(B∩C)+P(A∩B∩C)$ | — |

In [ ]:
# Numerically verify all propositions
# Using the fair die
P = die.P  # shorthand
A = {2, 4, 6}    # even
B = {4, 5, 6}    # > 3
Ac = S_die - A

print("=" * 55)
print("Verification of Probability Propositions (Fair Die)")
print("=" * 55)

# 1. Complement
p_Ac_formula = 1 - P(A)
p_Ac_direct  = P(Ac)
print(f"\n1. Complement: P(Aᶜ) = 1 - P(A)")
print(f"   P(A) = P(even) = {P(A):.4f}")
print(f"   1-P(A) = {p_Ac_formula:.4f},  P(Aᶜ) = {p_Ac_direct:.4f}  → {'✓' if abs(p_Ac_formula - p_Ac_direct) < 1e-9 else '✗'}")

# 2. Empty set
print(f"\n2. P(∅) = {P(set()):.4f}  → {'✓' if P(set()) == 0 else '✗'}")

# 3. Difference rule (A={2,4} ⊂ B={2,4,6})
A2 = {2, 4};  B2 = {2, 4, 6}  # A2 ⊆ B2
print(f"\n3. Difference: P(B∩Aᶜ) = P(B)-P(A), A⊆B")
print(f"   A={sorted(A2)}, B={sorted(B2)} → A⊆B: {A2.issubset(B2)}")
lhs = P(B2 & (S_die - A2))
rhs = P(B2) - P(A2)
print(f"   P(B∩Aᶜ) = {lhs:.4f},  P(B)-P(A) = {rhs:.4f}  → {'✓' if abs(lhs-rhs)<1e-9 else '✗'}")

# 4. Monotonicity
print(f"\n4. Monotonicity: P(B)≥P(A) if A⊆B")
print(f"   P(A={sorted(A2)}) = {P(A2):.4f} ≤ P(B={sorted(B2)}) = {P(B2):.4f}  → {'✓' if P(B2)>=P(A2) else '✗'}")

# 5. Inclusion-Exclusion (2 events)
print(f"\n5. Inclusion-Exclusion (2 events): P(A∪B) = P(A)+P(B)-P(A∩B)")
direct  = P(A | B)
formula = P(A) + P(B) - P(A & B)
print(f"   A={sorted(A)}, B={sorted(B)}")
print(f"   P(A∪B) directly = {direct:.4f}")
print(f"   P(A)+P(B)-P(A∩B) = {P(A):.4f}+{P(B):.4f}-{P(A&B):.4f} = {formula:.4f}  → {'✓' if abs(direct-formula)<1e-9 else '✗'}")

In [ ]:
# Textbook example: Probability class student
print("=" * 55)
print("Inclusion-Exclusion Example — Probability Class")
print("=" * 55)

p_A  = 0.63   # P(Eastern time zone)
p_B  = 0.41   # P(Senior)
p_AB = 0.31   # P(Both)

p_AuB = p_A + p_B - p_AB

print(f"P(A) = P(Eastern time zone) = {p_A}")
print(f"P(B) = P(Senior)            = {p_B}")
print(f"P(A∩B) = P(Both)            = {p_AB}")
print(f"\nP(A∪B) = {p_A} + {p_B} - {p_AB} = {p_AuB}")

# Additional computations
p_Ac      = 1 - p_A
p_Bc      = 1 - p_B
p_Ac_Bc   = 1 - p_AuB   # DeMorgan
p_only_A  = p_A - p_AB
p_only_B  = p_B - p_AB

print(f"\nAdditional computations:")
print(f"  P(Aᶜ) = {p_Ac:.2f}  (not in Eastern time zone)")
print(f"  P(Bᶜ) = {p_Bc:.2f}  (not a Senior)")
print(f"  P(Neither A nor B) = 1-P(A∪B) = {p_Ac_Bc:.2f}")
print(f"  P(Only A) = {p_only_A:.2f}")
print(f"  P(Only B) = {p_only_B:.2f}")
print(f"  Total: {p_only_A:.2f}+{p_only_B:.2f}+{p_AB:.2f}+{p_Ac_Bc:.2f} = {p_only_A+p_only_B+p_AB+p_Ac_Bc:.2f}  ✓")

In [ ]:
# Inclusion-Exclusion visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: Student example Venn diagram
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 7); ax.set_aspect('equal'); ax.axis('off')
ax.set_title('Student Example — Inclusion-Exclusion', fontsize=11, fontweight='bold')

rect = plt.Rectangle((0.2, 0.5), 9.6, 6, fill=False, edgecolor='black', linewidth=2)
ax.add_patch(rect)
cA = Circle((3.5, 3.5), 2.4, color='#92c5de', alpha=0.6, zorder=2)
cB = Circle((6.5, 3.5), 2.4, color='#f4a582', alpha=0.6, zorder=2)
ax.add_patch(cA); ax.add_patch(cB)
ca = Circle((3.5,3.5),2.4,fill=False,ec='#2166ac',lw=2,zorder=3)
cb = Circle((6.5,3.5),2.4,fill=False,ec='#ca0020',lw=2,zorder=3)
ax.add_patch(ca); ax.add_patch(cb)

ax.text(2.3, 3.5, f'A only\n{p_only_A:.2f}', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(5.0, 3.5, f'A∩B\n{p_AB:.2f}', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(7.7, 3.5, f'B only\n{p_only_B:.2f}', ha='center', fontsize=9, fontweight='bold', zorder=4)
ax.text(1.2, 1.2, f'Neither\n{p_Ac_Bc:.2f}', ha='center', fontsize=9, zorder=4)
ax.text(2.0, 5.8, 'A=Eastern\nTime Zone', ha='center', fontsize=8, color='#2166ac')
ax.text(8.0, 5.8, 'B=Senior', ha='center', fontsize=8, color='#ca0020')
ax.text(9.5, 6.3, 'S', fontsize=13, fontweight='bold')
ax.text(5, 0.2, f'P(A∪B) = {p_AuB:.2f}', ha='center', fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow'))

# Right: 3-event Inclusion-Exclusion
ax2 = axes[1]
ax2.set_xlim(0, 10); ax2.set_ylim(0, 7.5); ax2.set_aspect('equal'); ax2.axis('off')
ax2.set_title('3 Events — Inclusion-Exclusion\nP(A∪B∪C)', fontsize=11, fontweight='bold')

rect2 = plt.Rectangle((0.2, 0.5), 9.6, 6.5, fill=False, edgecolor='black', linewidth=2)
ax2.add_patch(rect2)

cA2 = Circle((4.0, 4.2), 2.0, color='#92c5de', alpha=0.45, zorder=2)
cB2 = Circle((6.0, 4.2), 2.0, color='#f4a582', alpha=0.45, zorder=2)
cC2 = Circle((5.0, 2.4), 2.0, color='#b2df8a', alpha=0.45, zorder=2)
for c in [cA2, cB2, cC2]: ax2.add_patch(c)
for (cx,cy,r,col) in [(4.0,4.2,2.0,'#2166ac'),(6.0,4.2,2.0,'#ca0020'),(5.0,2.4,2.0,'#33a02c')]:
    ax2.add_patch(Circle((cx,cy),r,fill=False,ec=col,lw=2,zorder=3))

ax2.text(3.0, 5.2, 'A', fontsize=13, fontweight='bold', color='#2166ac', ha='center')
ax2.text(7.0, 5.2, 'B', fontsize=13, fontweight='bold', color='#ca0020', ha='center')
ax2.text(5.0, 1.2, 'C', fontsize=13, fontweight='bold', color='#33a02c', ha='center')
ax2.text(5, 3.4, 'A∩B∩C', ha='center', fontsize=7, fontweight='bold', zorder=4)
ax2.text(9.5, 6.8, 'S', fontsize=13, fontweight='bold')
ax2.text(5, 0.2,
         'P(A∪B∪C) = P(A)+P(B)+P(C)\n  −P(A∩B)−P(A∩C)−P(B∩C)+P(A∩B∩C)',
         ha='center', fontsize=8,
         bbox=dict(boxstyle='round', facecolor='lightyellow'))

plt.tight_layout()
plt.show()

---
## 6. Equally Likely Sample Spaces

If the sample space contains $N$ equally likely outcomes:

$$P(\{i\}) = \frac{1}{N} \qquad \Rightarrow \qquad P(E) = \frac{\#(E)}{\#(S)}$$

In [ ]:
def uniform_probability(S, E):
    """P(E) = |E|/|S| for an equally likely sample space"""
    return len(E) / len(S)

print("=" * 50)
print("Equally Likely Sample Space Examples")
print("=" * 50)

# 1. Die: even number
S1 = {1,2,3,4,5,6}
E1 = {2,4,6}
p1 = uniform_probability(S1, E1)
print(f"\n1. Die: Even number")
print(f"   S={sorted(S1)}, E={sorted(E1)}")
print(f"   P(E) = {len(E1)}/{len(S1)} = {p1:.4f} = {Fraction(len(E1), len(S1))}")

# 2. Two children: at least one boy
S2 = {'BB', 'BG', 'GB', 'GG'}
E2 = {'BB', 'BG', 'GB'}  # at least one boy
p2 = uniform_probability(S2, E2)
print(f"\n2. Two children: at least one boy")
print(f"   S={sorted(S2)}, E={sorted(E2)}")
print(f"   P(E) = {len(E2)}/{len(S2)} = {p2:.4f} = {Fraction(len(E2), len(S2))}")

# 3. Coin tosses: exactly 2 heads
S3 = {''.join(s) for s in itertools.product('HT', repeat=3)}
E3 = {s for s in S3 if s.count('H') == 2}
p3 = uniform_probability(S3, E3)
print(f"\n3. 3 coin tosses: exactly 2 heads")
print(f"   |S| = {len(S3)}, E = {sorted(E3)}")
print(f"   P(E) = {len(E3)}/{len(S3)} = {p3:.4f} = {Fraction(len(E3), len(S3))}")

# 4. Poker: at least 1 ace
total_hand = math.comb(52, 5)
at_least_1_ace = total_hand - math.comb(48, 5)  # at least 1 = total - (none)
p4 = at_least_1_ace / total_hand
print(f"\n4. Poker (52 cards, draw 5): at least 1 ace")
print(f"   C(52,5) = {total_hand:,}")
print(f"   P(at least 1 ace) = 1 - P(no ace) = 1 - C(48,5)/C(52,5)")
print(f"                     = {p4:.5f} ≈ {p4*100:.2f}%")

---
## 7. The Birthday Problem 🎂

The probability that no two people among $n$ share a birthday:

$$P(\text{no match}) = \frac{365!}{(365-n)! \cdot 365^n} = \frac{365}{365} \cdot \frac{364}{365} \cdots \frac{365-n+1}{365}$$

$$P(\text{at least one match}) = 1 - P(\text{no match})$$

**Surprising result:** Even for $n = 23$, $P(\text{match}) > 0.5$!

In [ ]:
def p_no_match(n, days=365):
    """Probability that no two people among n share a birthday"""
    if n > days:
        return 0.0
    p = 1.0
    for k in range(n):
        p *= (days - k) / days
    return p

def p_at_least_one_match(n, days=365):
    return 1 - p_no_match(n, days)

# Table
print(f"{'n':>4} | {'P(no match)':>18} | {'P(at least 1 match)':>21}")
print("-" * 50)
notable_n = [5, 10, 15, 20, 23, 25, 30, 40, 50, 60, 70]
for n in notable_n:
    p_no  = p_no_match(n)
    p_yes = p_at_least_one_match(n)
    marker = " ← 50% threshold!" if n == 23 else ""
    print(f"{n:>4} | {p_no:>18.6f} | {p_yes:>18.6f}{marker}")

In [ ]:
# Birthday problem — visualization
n_values  = np.arange(1, 80)
p_no_arr  = [p_no_match(n) for n in n_values]
p_yes_arr = [p_at_least_one_match(n) for n in n_values]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: probability curves
ax = axes[0]
ax.plot(n_values, p_no_arr,  'b-', linewidth=2.5, label='P(no match)')
ax.plot(n_values, p_yes_arr, 'r-', linewidth=2.5, label='P(at least 1 match)')
ax.axhline(0.5, color='green', linestyle='--', linewidth=1.5, alpha=0.7, label='P = 0.5')
ax.axvline(23, color='purple', linestyle=':', linewidth=2, alpha=0.8, label='n = 23')

# Mark n=23
ax.scatter([23], [p_at_least_one_match(23)], color='red', s=120, zorder=5)
ax.annotate(f'n=23\nP={p_at_least_one_match(23):.3f}',
            xy=(23, p_at_least_one_match(23)),
            xytext=(35, 0.45),
            arrowprops=dict(arrowstyle='->', color='black'),
            fontsize=10, bbox=dict(boxstyle='round', facecolor='lightyellow'))

ax.set_xlabel('Number of people (n)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title('The Birthday Problem', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)

# Right: Monte Carlo validation
ax2 = axes[1]

def monte_carlo_birthday(n, n_trials=10000):
    matches = 0
    for _ in range(n_trials):
        birthdays = np.random.randint(1, 366, size=n)
        if len(birthdays) != len(set(birthdays)):
            matches += 1
    return matches / n_trials

np.random.seed(42)
n_test      = [5, 10, 15, 20, 23, 30, 40, 50]
mc_results  = [monte_carlo_birthday(n, 8000) for n in n_test]
theo_results = [p_at_least_one_match(n) for n in n_test]

x_pos = np.arange(len(n_test))
width = 0.35
ax2.bar(x_pos - width/2, theo_results, width, label='Theoretical',
        color='#4C72B0', alpha=0.85, edgecolor='black', linewidth=0.5)
ax2.bar(x_pos + width/2, mc_results, width, label='Monte Carlo (8000 trials)',
        color='#DD8452', alpha=0.85, edgecolor='black', linewidth=0.5)
ax2.axhline(0.5, color='green', linestyle='--', linewidth=1.5, alpha=0.7)
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'n={n}' for n in n_test], rotation=30, ha='right')
ax2.set_ylabel('P(at least 1 match)', fontsize=11)
ax2.set_title('Theoretical vs Monte Carlo Validation', fontsize=12, fontweight='bold')
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis='y')
ax2.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

print("\nComparison:")
print(f"{'n':>4} | {'Theoretical':>12} | {'Monte Carlo':>12} | {'Difference':>10}")
print("-" * 46)
for n, t, mc in zip(n_test, theo_results, mc_results):
    print(f"{n:>4} | {t:>12.5f} | {mc:>12.5f} | {abs(t-mc):>10.5f}")

---
## 8. Interpretations of Probability

**Frequentist:** $P(A) = \lim_{n \to \infty} \dfrac{\text{number of occurrences of A}}{n}$

**Bayesian:** $P(A)$ represents a personal degree of belief; it is updated with new data via **Bayes' Rule**.

In [ ]:
# Frequentist interpretation — coin toss simulation
np.random.seed(42)
N_max  = 10_000
tosses = np.random.randint(0, 2, size=N_max)  # 0=Tails, 1=Heads

# Cumulative proportion of heads
n_seq        = np.arange(1, N_max + 1)
cum_heads    = np.cumsum(tosses == 1) / n_seq

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: convergence
ax = axes[0]
ax.semilogx(n_seq, cum_heads, color='#2166ac', linewidth=1.5, alpha=0.9,
            label='Observed proportion')
ax.axhline(0.5, color='red', linestyle='--', linewidth=2, label='Theoretical P(Heads) = 0.5')
ax.fill_between(n_seq,
                0.5 - 1/np.sqrt(n_seq),
                0.5 + 1/np.sqrt(n_seq),
                alpha=0.15, color='red', label='±1/√n confidence band')
ax.set_xlabel('Number of tosses (log scale)', fontsize=11)
ax.set_ylabel('Cumulative Proportion of Heads', fontsize=11)
ax.set_title('Frequentist Interpretation — Law of Large Numbers',
             fontsize=12, fontweight='bold')
ax.set_ylim(0.3, 0.7)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Right: Several independent runs
ax2 = axes[1]
np.random.seed(0)
for trial_no in range(8):
    tosses_i = np.random.randint(0, 2, size=N_max)
    cum_i    = np.cumsum(tosses_i == 1) / n_seq
    ax2.semilogx(n_seq, cum_i, alpha=0.5, linewidth=1)
ax2.axhline(0.5, color='black', linestyle='--', linewidth=2.5, label='P = 0.5')
ax2.set_xlabel('Number of tosses (log scale)', fontsize=11)
ax2.set_ylabel('Cumulative Proportion of Heads', fontsize=11)
ax2.set_title("8 Independent Simulations — All Converge to P=0.5",
              fontsize=12, fontweight='bold')
ax2.set_ylim(0.3, 0.7)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Final value (10000 tosses): P̂(Heads) = {cum_heads[-1]:.5f}  (theoretical: 0.5)")

---
## 9. Summary

In [ ]:
print("=" * 65)
print("CHAPTER 2 SUMMARY — Axioms of Probability")
print("=" * 65)

summary = [
    ("Sample Space S",       "Set of all possible outcomes"),
    ("Event E",              "E ⊆ S (subset of the sample space)"),
    ("Intersection A∩B",    "Both A and B"),
    ("Union A∪B",           "A or B (at least one)"),
    ("Complement Aᶜ",       "A does not occur = S\\A"),
    ("Mutually exclusive",  "A∩B = ∅"),
    ("DeMorgan 1",          "(A∪B)ᶜ = Aᶜ∩Bᶜ"),
    ("DeMorgan 2",          "(A∩B)ᶜ = Aᶜ∪Bᶜ"),
    ("Axiom 1",             "0 ≤ P(E) ≤ 1"),
    ("Axiom 2",             "P(S) = 1"),
    ("Axiom 3",             "P(∪Eᵢ) = ΣP(Eᵢ)  (mutually exclusive events)"),
    ("Complement rule",     "P(Aᶜ) = 1 − P(A)"),
    ("Inclusion-Exclusion", "P(A∪B) = P(A)+P(B)−P(A∩B)"),
    ("Equally likely",      "P(E) = |E| / |S|"),
    ("Birthday problem",    "n=23: P(match)>0.5  (surprising!)"),
]

print(f"{'Concept':<24} | {'Description / Formula'}")
print("-" * 65)
for concept, description in summary:
    print(f"{concept:<24} | {description}")
print("=" * 65)

---
## 10. Exercises

**Problem 1:** In a single die roll, what is $P(\text{greater than 5 or even})$?

**Problem 2:** In a class of 30 students, what is the probability that no two students share a birthday?

**Problem 3:** Given $P(A) = 0.4$, $P(B) = 0.5$, $P(A \cap B) = 0.2$, find $P(A^c \cup B^c)$.

**Problem 4:** When 5 cards are drawn from a standard 52-card deck, what is the probability of getting at least one heart?

In [ ]:
# ANSWERS
print("PROBLEM 1: P(greater than 5 OR even)")
S1 = {1,2,3,4,5,6}
A1 = {6}        # greater than 5
B1 = {2,4,6}    # even
p_s1 = (len(A1) + len(B1) - len(A1&B1)) / len(S1)
print(f"  P(A∪B) = P({sorted(A1)}∪{sorted(B1)}) = {p_s1:.4f} = {Fraction(len(A1)+len(B1)-len(A1&B1), len(S1))}")

print()
print("PROBLEM 2: No shared birthday among 30 students")
p_s2 = p_no_match(30)
print(f"  P(no match, n=30) = {p_s2:.6f} ≈ {p_s2*100:.2f}%")
print(f"  P(at least 1 match) = {1-p_s2:.6f} ≈ {(1-p_s2)*100:.2f}%")

print()
print("PROBLEM 3: P(Aᶜ∪Bᶜ) = P((A∩B)ᶜ)  [DeMorgan]")
pA, pB, pAB = 0.4, 0.5, 0.2
p_s3 = 1 - pAB  # DeMorgan: (A∩B)ᶜ
print(f"  P(Aᶜ∪Bᶜ) = P((A∩B)ᶜ) = 1 - P(A∩B) = 1 - {pAB} = {p_s3}")

print()
print("PROBLEM 4: At least 1 heart when drawing 5 cards from 52")
total = math.comb(52, 5)
no_hearts = math.comb(39, 5)  # 52-13=39 non-heart cards
p_s4 = 1 - no_hearts / total
print(f"  P(at least 1 heart) = 1 - C(39,5)/C(52,5)")
print(f"                      = 1 - {no_hearts:,}/{total:,}")
print(f"                      = {p_s4:.5f} ≈ {p_s4*100:.2f}%")